In [12]:
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ----------------------------
# Custom Dataset (Part 1)
# ----------------------------
class CustomDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        # Search for JPG images in the specified directory.
        self.image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
        print(f"Found {len(self.image_paths)} images in {image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load an image and convert it to RGB
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        # Apply transformations if provided (e.g., resize, to tensor)
        if self.transform:
            image = self.transform(image)
        # In our demo, the same image serves as both low-res input and high-res target.
        return {'low_res': image, 'high_res': image}

# ----------------------------
# Image Transformations
# ----------------------------
# Resize images to 299x299 and convert them to tensors.
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
])

# ----------------------------
# DataLoader Setup
# ----------------------------
# Set the directory where your resized images are stored.
resized_dir = '/kaggle/working/resized_images'
# Create an instance of the custom dataset.
train_dataset = CustomDataset(resized_dir, transform=transform)
# Create a DataLoader to iterate through the dataset.
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# ----------------------------
# Debug: Verify DataLoader Output
# ----------------------------
# Iterate over one batch and print the keys and shapes.
for batch in train_loader:
    print("Batch keys:", batch.keys())
    print("Low-res image batch shape:", batch['low_res'].shape)  # Expected: [4, 3, 299, 299]
    break

Found 1887 images in /kaggle/working/resized_images
Batch keys: dict_keys(['low_res', 'high_res'])
Low-res image batch shape: torch.Size([4, 3, 299, 299])
